In [19]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Charger les données
df = pd.read_csv("../data/Smart_Farming_Crop_Yield_2024.csv")

# Features
features = [
    'soil_moisture_%',
    'temperature_C',
    'humidity_%',
    'rainfall_mm',
    'sunlight_hours'
]

# Sélection
df_selected = df[features].copy()

# 🔥 NORMALISATION
scaler = StandardScaler()
df_scaled_array = scaler.fit_transform(df_selected)

# transformer en DataFrame
df_scaled = pd.DataFrame(df_scaled_array, columns=features)

# 🔥 SCORE INTELLIGENT
score = (
    (1 - df_scaled['soil_moisture_%']) * 0.4 +
    df_scaled['temperature_C'] * 0.2 +
    (1 - df_scaled['humidity_%']) * 0.15 +
    (1 - df_scaled['rainfall_mm']) * 0.15 +
    df_scaled['sunlight_hours'] * 0.1
)

# 🔥 TARGET
df["irrigation_needed"] = (score > 0.5).astype(int)

# Vérification
print(df[['soil_moisture_%', 'irrigation_needed']].head())
print(df["irrigation_needed"].value_counts())

   soil_moisture_%  irrigation_needed
0            35.95                  0
1            19.74                  1
2            29.32                  1
3            17.33                  1
4            19.37                  1
irrigation_needed
1    324
0    176
Name: count, dtype: int64


In [20]:
Q1 = df_selected.quantile(0.25)
Q3 = df_selected.quantile(0.75)
IQR = Q3 - Q1
    
df_selected = df_selected[~((df_selected < (Q1 - 1.5 * IQR)) | (df_selected > (Q3 + 1.5 * IQR))).any(axis=1)]

In [21]:
df

,farm_id,region,crop_type,soil_moisture_%,soil_pH,temperature_C,rainfall_mm,humidity_%,sunlight_hours,irrigation_type,...,harvest_date,total_days,yield_kg_per_hectare,sensor_id,timestamp,latitude,longitude,NDVI_index,crop_disease_status,irrigation_needed
0,FARM0001,North India,Wheat,35.95,5.99,17.79,75.62,77.03,7.27,NaN,...,2024-05-09,122,4408.07,SENS0001,2024-03-19,14.970941,82.997689,0.63,Mild,0
1,FARM0002,South USA,Soybean,19.74,7.24,30.18,89.91,61.13,5.67,Sprinkler,...,2024-05-26,112,5389.98,SENS0002,2024-04-21,16.613022,70.869009,0.58,NaN,1
2,FARM0003,South USA,Wheat,29.32,7.16,27.37,265.43,68.87,8.23,Drip,...,2024-06-26,144,2931.16,SENS0003,2024-02-28,19.503156,79.068206,0.80,Mild,1
3,FARM0004,Central USA,Maize,17.33,6.03,33.73,212.01,70.46,5.03,Sprinkler,...,2024-07-04,134,4227.80,SENS0004,2024-05-14,31.071298,85.519998,0.44,NaN,1
4,FARM0005,Central USA,Cotton,19.37,5.92,33.86,269.09,55.73,7.93,NaN,...,2024-05-20,105,4979.96,SENS0005,2024-04-13,16.568540,81.691720,0.84,Severe,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,FARM0496,Central USA,Rice,42.85,6.70,30.85,52.35,79.58,7.25,Manual,...,2024-06-02,138,4251.40,SENS0496,2024-05-08,30.386623,76.147700,0.59,Mild,0
496,FARM0497,North India,Soybean,34.22,6.75,17.46,256.23,45.14,5.78,NaN,...,2024-04-14,104,3708.54,SENS0497,2024-01-19,18.832748,75.736924,0.85,Severe,0
497,FARM0498,North India,Cotton,15.93,5.72,17.03,288.96,57.87,7.69,Drip,...,2024-05-09,128,2604.41,SENS0498,2024-04-20,23.262016,81.992230,0.71,Mild,1
498,FARM0499,Central USA,Soybean,38.61,6.20,17.08,279.06,73.09,9.60,Drip,...,2024-06-04,131,2586.36,SENS0499,2024-03-02,19.764989,84.426869,0.77,Severe,0


In [22]:

import numpy as np
# 🔹 X = features
X = df_selected

# 🔹 y = target
y = df["irrigation_needed"]
# Séparer les features et la cible

# Diviser en train et test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Normaliser les features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Taille des données d'entraînement :", X_train_scaled.shape)
print("Taille des données de test :", X_test_scaled.shape)
print("Distribution de la cible en train :", np.bincount(y_train))
print("Distribution de la cible en test :", np.bincount(y_test))

Taille des données d'entraînement : (400, 5)
Taille des données de test : (100, 5)
Distribution de la cible en train : [141 259]
Distribution de la cible en test : [35 65]


In [25]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)
import joblib
import os

# Entraîner un modèle simple
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

# Sauvegarder le modèle et le scaler pour l'API
os.makedirs(os.path.join("..", "models"), exist_ok=True)
joblib.dump(model, os.path.join("..", "models", "model.pkl"))
joblib.dump(scaler, os.path.join("..", "models", "scaler.pkl"))
print("Modèle et scaler sauvegardés dans ../models/")

Accuracy: 0.95
Precision: 0.9411764705882353
Recall: 0.9846153846153847
F1-score: 0.9624060150375939

Confusion Matrix:
 [[31  4]
 [ 1 64]]
ROC-AUC: 0.9898901098901098
Modèle et scaler sauvegardés dans ../models/


In [26]:
print(df["irrigation_needed"].value_counts(normalize=True))

irrigation_needed
1    0.648
0    0.352
Name: proportion, dtype: float64


In [29]:
joblib.dump(model, "../models/model.pkl")
joblib.dump(scaler, "../models/scaler.pkl")

['../models/scaler.pkl']

In [31]:
import numpy as np

input_data = np.array([[
    20,   # soil_moisture_%
    35,   # temperature_C
    50,   # humidity_%
    0,    # rainfall_mm
    10    # sunlight_hours
]])

# scaler + prediction
input_scaled = scaler.transform(input_data)
result = model.predict(input_scaled)

print(result)

c:\Users\DELL\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


[1]


In [33]:
input_scaled = scaler.transform(input_data)
result = model.predict(input_scaled)
print(result)

c:\Users\DELL\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


[1]
